# Create Novo Nordisk Foundation GRANTS Awards (GRANT PATTERN, method-3 search API)

Novo Nordisk Foundation **grant-funding** side — the foundation's open grant-recipient (funded-projects) database at novonordiskfonden.dk. One of the world's largest biomedical-research funders by endowment (DKK 100+ billion); awarded ~DKK 10 billion in 2024.

**Prerequisites:**
- Run `scripts/local/novo_nordisk_to_s3.py` first (uploads the parquet to S3).

**Data source:** the grant-recipient search at `/en/grant-recipient/` is backed by an admin-ajax `content_api` endpoint (`type=grant_recipients`) that returns the canonical funded-projects list. Reported corpus: **8,987 awarded grants** (`data-total-posts`). Method-3 (search/index API) on the runbook ingest ladder.

Per-grant fields published: recipient/PI name + role, host institution (location), project title (description), **amount in DKK**, and grant year. No per-grant detail page, no native grant ID, no start/end dates beyond the single year.

**S3 location:** `s3a://openalex-ingest/awards/novo_nordisk/novo_nordisk_projects.parquet`

**Funder (Path A — F4320\* Crossref-registered, in `openalex.common.funder`):**
- funder_id: 4320325957
- display_name: "Novo Nordisk Fonden"
- country: DK
- ROR: none
- DOI: 10.13039/501100009708

**De-dup vs the PRIZE side (CRITICAL):** the foundation's PRIZE recipients are already ingested at **priority 119**, provenance `novo_nordisk_fonden_prizes` (`CreateNovoNordiskFondenAwards.ipynb`). This GRANT ingest is fully disjoint from it: different provenance (`novo_nordisk_fonden_grants`), different priority (208), and a different synthetic-ID namespace (`nnf-grant-*` here vs `nnf-{slug}` for prizes). The two never collide on `funder_award_id`, so prizes are not double-counted as grants.

**Amount/currency:** POPULATED in **DKK** on essentially every row (parsed from the Danish-formatted "DKK 16.151.127"). **NOT a §6.7 waiver** — NNF publishes the amount per grant.

**PI vs institution:** a row is institution-level when the recipient role (title) is empty AND the recipient name either equals the host institution (e.g. "John Innes Centre") or looks like an organization (NGO/university/institute/foundation — e.g. "Røde Kors", "UNICEF Danmark", "World Diabetes Foundation", "Statens Serum Institut"). Those carry NULL `lead_investigator` names but keep the org as `affiliation.name` (runbook §6.4a — an institution must never appear as a PI given/family name). Other rows are people — `lead_given_name`/`lead_family_name` were split in Python (runbook §2.4.1 `split_name`). Recipient institutions are international, so `affiliation.country` is left NULL.

**funder_scheme:** the grant programme/scheme, enriched in the scraper from the search page's `category` filter (nullable — rows with no category match keep NULL).

**Provenance:** `novo_nordisk_fonden_grants`.

**Priority:** 208.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.novo_nordisk_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/novo_nordisk/novo_nordisk_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.novo_nordisk_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.novo_nordisk_raw;

## Step 1.5: Inspect Raw Data, Money Scan, Native Key

NNF publishes amounts structurally in DKK — expect very high pct_amount.

In [ ]:
%sql
-- Money-flavored columns (runbook §1.6 discovery scan)
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards' AND table_name = 'novo_nordisk_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';

In [ ]:
%sql
-- Currency-flavored columns
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards' AND table_name = 'novo_nordisk_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';

In [ ]:
%sql
SELECT funder_award_id, display_name, owner_name_raw, owner_title_raw,
       lead_given_name, lead_family_name, is_institutional, institution,
       amount, amount_raw, currency, award_year, funder_scheme
FROM openalex.awards.novo_nordisk_raw LIMIT 10;

In [ ]:
%sql
-- Amount sanity: a real grant distribution sits in the thousands-millions of DKK.
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_dkk,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_dkk,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_dkk,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)) / 1e9, 2) AS total_dkk_billions,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.novo_nordisk_raw;

In [ ]:
%sql
-- Native-key collision check: synthetic funder_award_id must be unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.novo_nordisk_raw;

In [ ]:
%sql
SELECT award_year, COUNT(*) as grants
FROM openalex.awards.novo_nordisk_raw
WHERE award_year IS NOT NULL
GROUP BY award_year ORDER BY award_year DESC LIMIT 30;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as grants,
       ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6, 1) AS total_dkk_millions
FROM openalex.awards.novo_nordisk_raw
GROUP BY funder_scheme ORDER BY grants DESC LIMIT 25;

In [ ]:
%sql
-- PI vs institution split + top institutions
SELECT is_institutional, COUNT(*) as cnt
FROM openalex.awards.novo_nordisk_raw GROUP BY is_institutional;

## Step 1.6: Fail-fast — Verify the Novo Nordisk Fonden Funder Row Exists (Path A)

In [ ]:
%sql
-- Path A (F4320*): MUST return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320325957;

In [ ]:
%sql
SELECT assert_true(
    (SELECT COUNT(*) FROM openalex.common.funder WHERE funder_id = 4320325957) = 1,
    'Novo Nordisk Fonden (F4320325957) missing from openalex.common.funder — STOP'
);

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.novo_nordisk_awards
USING delta
AS
WITH
nnf_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320325957
),
raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(award_year AS INT) AS parsed_year,
        CASE
            WHEN TRY_CAST(award_year AS INT) IS NOT NULL
            THEN TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd')
            ELSE NULL
        END AS parsed_start_date
    FROM openalex.awards.novo_nordisk_raw
    WHERE display_name IS NOT NULL
      AND TRIM(display_name) != ''
      AND funder_award_id IS NOT NULL
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.display_name as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        g.funder_award_id,

        g.parsed_amount as amount,
        g.currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,
        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'novo_nordisk_fonden_grants' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        CASE
            WHEN g.institution IS NULL OR TRIM(g.institution) = '' THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                NULLIF(TRIM(g.lead_given_name), '') as given_name,
                NULLIF(TRIM(g.lead_family_name), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(g.institution), '') as name,
                    CAST(NULL AS STRING) as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        CAST(NULL AS STRING) as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN nnf_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert Into `openalex_awards_raw` With Priority 208

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'novo_nordisk_fonden_grants' AND priority = 208;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    208 as priority
FROM openalex.awards.novo_nordisk_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.novo_nordisk_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.novo_nordisk_awards;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(funder_scheme) as has_scheme,
    COUNT(lead_investigator) as has_lead,
    COUNT(lead_investigator.family_name) as has_pi_name,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount,
    ROUND(try_divide(COUNT(funder_scheme), COUNT(*)) * 100.0, 1) as pct_scheme
FROM openalex.awards.novo_nordisk_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme,
       amount, currency,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family,
       lead_investigator.affiliation.name AS pi_affiliation
FROM openalex.awards.novo_nordisk_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a PI frequency check: top given+family combos must be a real long-tail,
-- and no institution name should appear as a PI name (org recipients are
-- classified institution-level upstream and have NULL PI names).
SELECT lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       COUNT(*) AS n
FROM openalex.awards.novo_nordisk_awards
WHERE lead_investigator.family_name IS NOT NULL
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.7 amount check. Expected: very high pct_amount, DKK only. NOT waived.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    ROUND(MIN(amount), 0) AS min_dkk,
    ROUND(MAX(amount), 0) AS max_dkk,
    ROUND(AVG(amount), 0) AS avg_dkk,
    ROUND(SUM(amount) / 1e9, 2) AS total_dkk_billions
FROM openalex.awards.novo_nordisk_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.novo_nordisk_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 25;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.novo_nordisk_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 30;

In [ ]:
%sql
-- Final identity check: no funder_award_id collisions after transform.
SELECT COUNT(*) AS total, COUNT(DISTINCT funder_award_id) AS distinct_ids
FROM openalex.awards.novo_nordisk_awards;

In [ ]:
%sql
-- De-dup sanity vs the PRIZE side: the two provenances must be disjoint.
SELECT provenance, priority, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE funder_id = 4320325957
GROUP BY provenance, priority ORDER BY priority;